# Part 2.1 — BloodMNIST: Synthetic Blood Cell Microscope Images with DCGAN

This notebook trains a **DCGAN** on the **BloodMNIST** dataset and evaluates the quality of generated blood cell microscope images.

## Notebook goals
1. Load and explore the BloodMNIST dataset.
2. Examine sample images and class balance.
3. Train a **DCGAN** to generate synthetic blood cell images.
4. Monitor generator and discriminator losses across epochs.
5. Generate fake microscope images and compare them with real images.
6. Compute a quantitative similarity metric using **Fréchet Inception Distance (FID)** if available.
7. Save report-ready plots and CSV outputs.

## Notes
- The uploaded `.npz` file already contains train, validation, and test splits.
- The GAN is trained on the **training images only**.
- The notebook is designed for **Google Colab** and can run on CPU or GPU, but GPU is strongly recommended.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS, PATHS, REPRODUCIBILITY, AND VISUAL STYLE
# ============================================================

from pathlib import Path
import random
import time
import warnings
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid

warnings.filterwarnings("ignore")

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

BASE_OUTPUT_DIR = Path("outputs") / "bloodmnist"
CSV_DIR = BASE_OUTPUT_DIR / "csv"
PLOTS_DIR = BASE_OUTPUT_DIR / "plots"

for folder in [BASE_OUTPUT_DIR, CSV_DIR, PLOTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"CSV outputs will be saved to: {CSV_DIR.resolve()}")
print(f"Plot outputs will be saved to: {PLOTS_DIR.resolve()}")

DATA_PATH = Path("bloodmnist (2).npz")
print(f"Expected dataset path: {DATA_PATH}")

COLOR_REAL = "#7A1F1F"
COLOR_FAKE = "#C97B63"
COLOR_ALT = "#6C8A3B"
COLOR_GRID = "#D9D2C3"
COLOR_TEXT = "#2F2A24"

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.facecolor"] = "#FAF7F2"
plt.rcParams["figure.facecolor"] = "#FFFFFF"
plt.rcParams["axes.edgecolor"] = "#B9AD99"
plt.rcParams["grid.color"] = COLOR_GRID
plt.rcParams["axes.labelcolor"] = COLOR_TEXT
plt.rcParams["xtick.color"] = COLOR_TEXT
plt.rcParams["ytick.color"] = COLOR_TEXT
plt.rcParams["text.color"] = COLOR_TEXT
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.grid"] = False

def save_plot(filename: str) -> None:
    filepath = PLOTS_DIR / filename
    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches="tight")
    print(f"Saved plot: {filepath}")

In [ ]:
# ============================================================
# CELL 2 — LOAD BLOODMNIST DATA
# ============================================================

data = np.load(DATA_PATH)

train_images = data["train_images"]
train_labels = data["train_labels"].reshape(-1)
val_images = data["val_images"]
val_labels = data["val_labels"].reshape(-1)
test_images = data["test_images"]
test_labels = data["test_labels"].reshape(-1)

dataset_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_images": [len(train_images), len(val_images), len(test_images)],
    "image_height": [train_images.shape[1], val_images.shape[1], test_images.shape[1]],
    "image_width": [train_images.shape[2], val_images.shape[2], test_images.shape[2]],
    "channels": [train_images.shape[3], val_images.shape[3], test_images.shape[3]],
})

dataset_summary.to_csv(CSV_DIR / "01_dataset_split_summary.csv", index=False)

display(dataset_summary)

In [ ]:
# ============================================================
# CELL 3 — CLASS DISTRIBUTION ANALYSIS
# ============================================================

CLASS_NAMES = {
    0: "basophil",
    1: "eosinophil",
    2: "erythroblast",
    3: "immature granulocytes",
    4: "lymphocyte",
    5: "monocyte",
    6: "neutrophil",
    7: "platelet",
}

def label_count_table(labels: np.ndarray, split_name: str) -> pd.DataFrame:
    counts = pd.Series(labels).value_counts().sort_index()
    df = pd.DataFrame({
        "class_id": counts.index,
        "class_name": [CLASS_NAMES[i] for i in counts.index],
        "count": counts.values,
        "split": split_name,
    })
    return df

train_label_counts = label_count_table(train_labels, "train")
val_label_counts = label_count_table(val_labels, "validation")
test_label_counts = label_count_table(test_labels, "test")

all_label_counts = pd.concat([train_label_counts, val_label_counts, test_label_counts], ignore_index=True)
all_label_counts.to_csv(CSV_DIR / "02_class_distribution_by_split.csv", index=False)

display(train_label_counts)

In [ ]:
# ============================================================
# CELL 4 — PLOT TRAINING CLASS DISTRIBUTION
# ============================================================

plt.figure(figsize=(10, 5))
plt.bar(train_label_counts["class_name"], train_label_counts["count"], color=COLOR_REAL)
plt.title("BloodMNIST Training Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=30, ha="right")
save_plot("01_train_class_distribution.png")
plt.show()

In [ ]:
# ============================================================
# CELL 5 — VISUALISE SAMPLE REAL BLOODMNIST IMAGES
# ============================================================

def show_sample_grid(images: np.ndarray, labels: np.ndarray, n_per_row: int = 4, n_rows: int = 4):
    total = n_per_row * n_rows
    indices = np.random.choice(len(images), total, replace=False)

    fig, axes = plt.subplots(n_rows, n_per_row, figsize=(10, 10))
    axes = axes.flatten()

    for ax, idx in zip(axes, indices):
        ax.imshow(images[idx])
        ax.set_title(CLASS_NAMES[int(labels[idx])], fontsize=9)
        ax.axis("off")

    save_plot("02_sample_real_bloodmnist_images.png")
    plt.show()

show_sample_grid(train_images, train_labels, n_per_row=4, n_rows=4)

In [ ]:
# ============================================================
# CELL 6 — CUSTOM DATASET AND PREPROCESSING
# ============================================================

class BloodMNISTDataset(Dataset):
    """
    Dataset wrapper that converts images to PyTorch tensors and
    normalises them to [-1, 1] for DCGAN training.
    """
    def __init__(self, images: np.ndarray, labels: np.ndarray):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].astype(np.float32) / 255.0
        image = (image * 2.0) - 1.0
        image = torch.tensor(image).permute(2, 0, 1)  # HWC -> CHW
        label = int(self.labels[idx])
        return image, label

BATCH_SIZE = 128

train_dataset = BloodMNISTDataset(train_images, train_labels)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Training batches per epoch: {len(train_loader)}")

In [ ]:
# ============================================================
# CELL 7 — DCGAN MODEL DEFINITIONS
# ============================================================

LATENT_DIM = 100
IMAGE_CHANNELS = 3
FEATURE_MAPS_GEN = 64
FEATURE_MAPS_DISC = 64

class Generator(nn.Module):
    """
    DCGAN generator for 28x28 RGB BloodMNIST images.

    The model upsamples from latent space into image space using
    transpose convolutions and Tanh output activation.
    """
    def __init__(self, latent_dim=LATENT_DIM, channels=IMAGE_CHANNELS, feature_maps=FEATURE_MAPS_GEN):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, feature_maps * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_maps * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps * 4, feature_maps * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps * 2, feature_maps, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps),
            nn.ReLU(True),

            nn.ConvTranspose2d(feature_maps, channels, 3, 1, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.model(z)
        return x[:, :, :28, :28]


class Discriminator(nn.Module):
    """
    DCGAN discriminator for 28x28 RGB BloodMNIST images.
    """
    def __init__(self, channels=IMAGE_CHANNELS, feature_maps=FEATURE_MAPS_DISC):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(channels, feature_maps, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps, feature_maps * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps * 2, feature_maps * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_maps * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feature_maps * 4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        out = self.model(x)
        return out.view(-1, 1)

generator = Generator().to(DEVICE)
discriminator = Discriminator().to(DEVICE)

print(generator)
print()
print(discriminator)

In [ ]:
# ============================================================
# CELL 8 — WEIGHT INITIALISATION
# ============================================================

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

generator.apply(weights_init)
discriminator.apply(weights_init)

In [ ]:
# ============================================================
# CELL 9 — TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 60
LEARNING_RATE = 0.0002
BETA1 = 0.5

criterion = nn.BCELoss()

optimizer_d = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))
optimizer_g = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(BETA1, 0.999))

fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, device=DEVICE)

print(f"NUM_EPOCHS = {NUM_EPOCHS}")
print(f"LEARNING_RATE = {LEARNING_RATE}")

In [ ]:
# ============================================================
# CELL 10 — HELPER FUNCTIONS FOR VISUALISATION
# ============================================================

def denormalise_images(x: torch.Tensor) -> torch.Tensor:
    """
    Convert images from [-1, 1] back to [0, 1].
    """
    return (x + 1.0) / 2.0

def save_generated_grid(generator: nn.Module, noise: torch.Tensor, epoch: int, filename: str):
    """
    Save a report-ready grid of generated images.
    """
    generator.eval()
    with torch.no_grad():
        fake = generator(noise).detach().cpu()
        fake = denormalise_images(fake)
        grid = make_grid(fake, nrow=8, padding=2)

    plt.figure(figsize=(8, 8))
    plt.imshow(np.transpose(grid.numpy(), (1, 2, 0)))
    plt.title(f"Generated BloodMNIST Images — Epoch {epoch}")
    plt.axis("off")
    save_plot(filename)
    plt.show()

In [ ]:
# ============================================================
# CELL 11 — TRAIN THE DCGAN
# ============================================================

history = []
epoch_sample_records = []

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    batches = 0

    for real_images, _ in train_loader:
        real_images = real_images.to(DEVICE)
        batch_size = real_images.size(0)

        real_labels = torch.ones((batch_size, 1), device=DEVICE)
        fake_labels = torch.zeros((batch_size, 1), device=DEVICE)

        # -----------------------------
        # Train discriminator
        # -----------------------------
        discriminator.zero_grad()

        output_real = discriminator(real_images)
        loss_d_real = criterion(output_real, real_labels)

        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=DEVICE)
        fake_images = generator(noise)

        output_fake = discriminator(fake_images.detach())
        loss_d_fake = criterion(output_fake, fake_labels)

        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        optimizer_d.step()

        # -----------------------------
        # Train generator
        # -----------------------------
        generator.zero_grad()

        output_fake_for_g = discriminator(fake_images)
        loss_g = criterion(output_fake_for_g, real_labels)
        loss_g.backward()
        optimizer_g.step()

        epoch_d_loss += loss_d.item()
        epoch_g_loss += loss_g.item()
        batches += 1

    avg_d_loss = epoch_d_loss / batches
    avg_g_loss = epoch_g_loss / batches

    history.append({
        "epoch": epoch,
        "loss_discriminator": avg_d_loss,
        "loss_generator": avg_g_loss,
    })

    if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
        print(
            f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
            f"Loss D: {avg_d_loss:.4f} | "
            f"Loss G: {avg_g_loss:.4f}"
        )

        save_generated_grid(
            generator=generator,
            noise=fixed_noise,
            epoch=epoch,
            filename=f"03_generated_samples_epoch_{epoch:03d}.png"
        )

training_time_seconds = time.time() - start_time
history_df = pd.DataFrame(history)
history_df.to_csv(CSV_DIR / "03_training_loss_history.csv", index=False)

print(f"Training completed in {training_time_seconds:.2f} seconds.")

In [ ]:
# ============================================================
# CELL 12 — PLOT TRAINING LOSSES
# ============================================================

plt.figure(figsize=(9, 5))
plt.plot(history_df["epoch"], history_df["loss_discriminator"], label="Discriminator Loss", color=COLOR_REAL, linewidth=2)
plt.plot(history_df["epoch"], history_df["loss_generator"], label="Generator Loss", color=COLOR_ALT, linewidth=2)
plt.title("BloodMNIST DCGAN Training Losses")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
save_plot("04_bloodmnist_dcgan_loss_curves.png")
plt.show()

In [ ]:
# ============================================================
# CELL 13 — FINAL REAL VS GENERATED IMAGE GRIDS
# ============================================================

def show_real_vs_generated(real_images_np: np.ndarray, generator: nn.Module, n_images: int = 64):
    """
    Compare real and generated image grids side by side.
    """
    real_idx = np.random.choice(len(real_images_np), n_images, replace=False)
    real_batch = torch.tensor(real_images_np[real_idx].astype(np.float32) / 255.0).permute(0, 3, 1, 2)

    noise = torch.randn(n_images, LATENT_DIM, 1, 1, device=DEVICE)
    generator.eval()
    with torch.no_grad():
        fake_batch = generator(noise).cpu()
        fake_batch = denormalise_images(fake_batch)

    real_grid = make_grid(real_batch, nrow=8, padding=2)
    fake_grid = make_grid(fake_batch, nrow=8, padding=2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(np.transpose(real_grid.numpy(), (1, 2, 0)))
    axes[0].set_title("Real BloodMNIST Images")
    axes[0].axis("off")

    axes[1].imshow(np.transpose(fake_grid.numpy(), (1, 2, 0)))
    axes[1].set_title("Generated BloodMNIST Images")
    axes[1].axis("off")

    save_plot("05_real_vs_generated_image_grids.png")
    plt.show()

show_real_vs_generated(train_images, generator, n_images=64)

In [ ]:
# ============================================================
# CELL 14 — OPTIONAL FID COMPUTATION
# ============================================================
"""
Compute Fréchet Inception Distance (FID) if torchmetrics is available.

If the package is not available, the notebook will continue gracefully.
"""

fid_result = None

try:
    from torchmetrics.image.fid import FrechetInceptionDistance

    fid_metric = FrechetInceptionDistance(feature=64, normalize=True).to(DEVICE)

    # Use a manageable evaluation subset.
    n_eval = min(1024, len(train_images))

    real_idx = np.random.choice(len(train_images), n_eval, replace=False)
    real_batch = train_images[real_idx].astype(np.float32) / 255.0
    real_batch = torch.tensor(real_batch).permute(0, 3, 1, 2).to(DEVICE)

    noise = torch.randn(n_eval, LATENT_DIM, 1, 1, device=DEVICE)
    with torch.no_grad():
        fake_batch = generator(noise)
        fake_batch = denormalise_images(fake_batch)

    # FID expects uint8-like images or normalized floats depending on setup.
    fid_metric.update(real_batch, real=True)
    fid_metric.update(fake_batch, real=False)
    fid_result = float(fid_metric.compute().cpu().item())

    print(f"FID score: {fid_result:.4f}")

except Exception as e:
    print("FID could not be computed in this environment.")
    print("Reason:", str(e))

In [ ]:
# ============================================================
# CELL 15 — REPORT-READY SUMMARY TABLE
# ============================================================

report_summary = pd.DataFrame({
    "item": [
        "train_images",
        "validation_images",
        "test_images",
        "image_height",
        "image_width",
        "channels",
        "n_classes",
        "batch_size",
        "latent_dim",
        "training_epochs",
        "training_time_seconds",
        "fid_score_if_available",
    ],
    "value": [
        int(len(train_images)),
        int(len(val_images)),
        int(len(test_images)),
        int(train_images.shape[1]),
        int(train_images.shape[2]),
        int(train_images.shape[3]),
        int(len(np.unique(train_labels))),
        int(BATCH_SIZE),
        int(LATENT_DIM),
        int(NUM_EPOCHS),
        round(training_time_seconds, 2),
        "not_computed" if fid_result is None else round(fid_result, 4),
    ],
})

report_summary.to_csv(CSV_DIR / "04_report_ready_summary_table.csv", index=False)
display(report_summary)

## Interpretation guide for the report

Use the outputs from this notebook to discuss the following:

### 1. Why DCGAN was used
BloodMNIST is an **image dataset**, so a convolutional GAN is more appropriate than a simple MLP GAN.  
Convolutional layers allow the model to capture local spatial structure such as cell edges, textures, and staining patterns.

### 2. What to say about training losses
- GAN losses are useful for monitoring training dynamics, but they do not perfectly reflect image quality.
- A visually better generator may not always correspond to a lower generator loss.
- Stable training is usually indicated by losses that avoid extreme collapse or explosion.

### 3. How to judge the generated images
Discuss whether the generated images:
- resemble real blood cell microscope images
- preserve circular or cellular structure
- contain obvious artefacts, blur, duplicated patterns, or unrealistic colors

### 4. How to use FID
If FID is available:
- lower values suggest that generated images are closer to the distribution of real images
- however, FID should be interpreted together with visual inspection

If FID is not available:
- make clear that the evaluation relied on qualitative comparison and training behaviour

### 5. Limitations
- BloodMNIST images are small (28×28), which limits fine-grained visual detail
- GANs can still suffer from mode collapse or repeated visual motifs
- medical realism should not be assumed purely from visual similarity

In [ ]:
# ============================================================
# CELL 16 — OPTIONAL LIST OF SAVED OUTPUT FILES
# ============================================================

print("Saved CSV files:")
for path in sorted(CSV_DIR.glob("*.csv")):
    print("-", path)

print("\nSaved plot files:")
for path in sorted(PLOTS_DIR.glob("*.png")):
    print("-", path)